# VMFCVD Inference Notebook
### Compatible with: vmfcvd_smote_3label_ham-et_RM (4-model FDM/DFDM, 5-model HAM)

## Why the old notebook failed — and how this one fixes it

When Python pickles an object it stores the **class name and module path** (e.g. `__main__.VMFCVD`).  
On unpickling it looks for that class in the current Python session.  
If the class is not defined yet → `AttributeError: Can't get attribute 'VMFCVD'`.

**Fix: every class that exists inside the .pkl files must be re-defined in this notebook  
before any `pickle.load()` is called.** Cell 01 does exactly that — it is a verbatim  
copy of all class definitions from the training notebook.

## Notebook layout

| Cell | What it does | Must run? |
|------|-------------|-----------|
| **01** | All imports + constants + class stubs (pickle prerequisite) | **Always first** |
| **02** | User configuration (paths, label, flow rate) | **Always** |
| **03** | Load checkpoints + resource-track loading stage | **Always** |
| **04** | Preprocessing helpers | **Always** |
| **05** | Metrics helpers (binary + 3-label) | **Always** |
| **Mode A** | Inference on a CSV file | On demand |
| **Mode B** | Inference on a single manually-specified row | On demand |
| **Mode C** | All three modes side-by-side comparison | On demand |
| **Mode D** | Batch predictor (large files only) | Optional |
| **Resource** | Print full DetailedResourceMonitor report | After any mode |


## Cell 01 — Foundation: imports, constants, and all class definitions

**Run this cell first, every time.** It defines every class that the pickle files reference.  
Without this cell, `pickle.load()` will raise `AttributeError: Can't get attribute 'VMFCVD'`.


In [1]:
# ── Standard library ─────────────────────────────────────────────────────────
import os, glob, pickle, warnings, math, time, threading, io, gc, sys
import numpy as np
import pandas as pd
from itertools import permutations
from typing import Dict, Any, List

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── sklearn (must match training environment) ─────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                               GradientBoostingClassifier, RandomForestClassifier,
                               ExtraTreesClassifier, HistGradientBoostingClassifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn import metrics

# ── psutil for resource monitoring ───────────────────────────────────────────
try:
    import psutil
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'psutil', '-q'])
    import psutil

# ── Constants (must match training notebook exactly) ─────────────────────────
# These are needed as default argument values in class __init__ signatures
# and referenced by lambdas inside VMFCVDVoter._BASE_MODELS_SPEC.
N_ESTIMATORS           = 100
RANDOM_STATE           = 42
USE_CLASS_WEIGHTS      = True
USE_SOFT_VOTING        = True
USE_CALIBRATION        = True
USE_SHAPLEY_WEIGHTS    = True
USE_EXTRATREES_HAM_ONLY = True
USE_EARLY_EXIT         = True
WARNING_PENALTY        = 0.05
FLOW_THRESHOLD_HIGH    = 1_000
FLOW_THRESHOLD_EXTREME = 5_000


# ═════════════════════════════════════════════════════════════════════════════
# DetailedResourceMonitor  (copied verbatim from training notebook cell 0)
# ═════════════════════════════════════════════════════════════════════════════

def print_system_info():
    vm = psutil.virtual_memory()
    print('\n' + '='*65)
    print('  SYSTEM INFORMATION')
    print('='*65)
    print(f'  CPU cores  (physical) : {psutil.cpu_count(logical=False)}')
    print(f'  CPU cores  (logical)  : {psutil.cpu_count(logical=True)}')
    print(f'  Total RAM             : {vm.total / 1024**3:.2f} GB')
    print(f'  Available RAM         : {vm.available / 1024**3:.2f} GB')
    print(f'  RAM utilisation       : {vm.percent:.1f}%')
    print(f'  Python                : {sys.version.split()[0]}')
    print(f'  Platform              : {sys.platform}')


class DetailedResourceMonitor:
    def __init__(self, poll_interval: float = 0.05):
        self.process       = psutil.Process()
        self.poll_interval = poll_interval
        self.stages: Dict[str, dict] = {}
        self.model_sizes: Dict[str, int] = {}
        self.inference_results: Dict[str, dict] = {}
        self._lock            = threading.Lock()
        self._active_stage    = None
        self._stop_event      = threading.Event()
        self._poll_thread     = threading.Thread(
            target=self._poll_worker, daemon=True, name='resource-monitor-poll')
        self._poll_thread.start()
        self._mem_samples:    Dict[str, List[float]] = {}
        self._cpu_samples:    Dict[str, List[float]] = {}
        self._thread_samples: Dict[str, List[int]]   = {}

    def _poll_worker(self):
        while not self._stop_event.is_set():
            with self._lock:
                stage = self._active_stage
            if stage:
                try:
                    mem_mb  = self.process.memory_info().rss / 1024 / 1024
                    cpu_pct = self.process.cpu_percent()
                    n_thr   = self.process.num_threads()
                    self._mem_samples[stage].append(mem_mb)
                    self._cpu_samples[stage].append(cpu_pct)
                    self._thread_samples[stage].append(n_thr)
                except Exception:
                    pass
            time.sleep(self.poll_interval)

    def start(self, stage: str):
        gc.collect()
        mem_now = self.process.memory_info().rss / 1024 / 1024
        self.stages[stage] = {
            'start_time'    : time.perf_counter(),
            'start_mem_mb'  : mem_now,
            'start_threads' : self.process.num_threads(),
        }
        self._mem_samples[stage]    = [mem_now]
        self._cpu_samples[stage]    = []
        self._thread_samples[stage] = [self.process.num_threads()]
        with self._lock:
            self._active_stage = stage

    def stop(self, stage: str) -> dict:
        with self._lock:
            if self._active_stage == stage:
                self._active_stage = None
        if stage not in self.stages:
            return {}
        s = self.stages[stage]
        s['elapsed_sec']      = time.perf_counter() - s['start_time']
        s['end_mem_mb']       = self.process.memory_info().rss / 1024 / 1024
        s['end_threads']      = self.process.num_threads()
        mem_s = self._mem_samples.get(stage, [s['end_mem_mb']])
        cpu_s = self._cpu_samples.get(stage, [0.0])
        thr_s = self._thread_samples.get(stage, [s['start_threads']])
        s['peak_mem_mb']      = float(max(mem_s))
        s['min_mem_mb']       = float(min(mem_s))
        s['mem_delta_mb']     = s['end_mem_mb'] - s['start_mem_mb']
        s['peak_increase_mb'] = s['peak_mem_mb'] - s['start_mem_mb']
        s['avg_cpu_pct']      = float(np.mean(cpu_s)) if cpu_s else 0.0
        s['max_cpu_pct']      = float(np.max(cpu_s))  if cpu_s else 0.0
        s['avg_threads']      = float(np.mean(thr_s))
        s['max_threads']      = int(np.max(thr_s))
        s['n_poll_samples']   = len(mem_s)
        return s

    def stop_all_polling(self):
        self._stop_event.set()
        self._poll_thread.join(timeout=2.0)

    def record_model_size(self, label: str, model) -> int:
        try:
            buf = io.BytesIO()
            pickle.dump(model, buf, protocol=4)
            nbytes = buf.tell()
        except Exception:
            nbytes = 0
        self.model_sizes[label] = nbytes
        return nbytes

    def record_inference(self, label: str, predict_fn, X,
                         n_repeats: int = 5, warmup: int = 1):
        n = len(X)
        for _ in range(warmup):
            predict_fn(X)
        stage = f'_infer_{label}'
        self.start(stage)
        times = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            predict_fn(X)
            times.append(time.perf_counter() - t0)
        self.stop(stage)
        mean_sec = float(np.mean(times))
        std_sec  = float(np.std(times))
        s = self.stages[stage]
        self.inference_results[label] = {
            'per_sample_us'    : mean_sec / n * 1e6,
            'per_sample_std_us': std_sec  / n * 1e6,
            'total_ms'         : mean_sec * 1000,
            'throughput_sps'   : n / mean_sec if mean_sec > 0 else 0,
            'peak_mem_mb'      : s['peak_mem_mb'],
            'mem_delta_mb'     : s['mem_delta_mb'],
            'avg_cpu_pct'      : s['avg_cpu_pct'],
            'max_cpu_pct'      : s['max_cpu_pct'],
            'max_threads'      : s['max_threads'],
            'n_samples'        : n,
            'n_repeats'        : n_repeats,
        }
        return self.inference_results[label]

    def summary_df(self, exclude_prefix: str = '_infer_') -> pd.DataFrame:
        rows = []
        for name, s in self.stages.items():
            if name.startswith(exclude_prefix) or 'elapsed_sec' not in s:
                continue
            rows.append({
                'Stage'         : name,
                'Time (s)'      : round(s['elapsed_sec'], 3),
                'Mem Δ (MB)'    : round(s['mem_delta_mb'], 2),
                'Peak Δ (MB)'   : round(s['peak_increase_mb'], 2),
                'Peak Mem (MB)' : round(s['peak_mem_mb'], 2),
                'Avg CPU (%)'   : round(s['avg_cpu_pct'], 1),
                'Max CPU (%)'   : round(s['max_cpu_pct'], 1),
                'Max Threads'   : s['max_threads'],
                'Poll samples'  : s['n_poll_samples'],
            })
        return pd.DataFrame(rows)

    def print_inference_report(self):
        if not self.inference_results:
            print('  No inference results recorded yet.')
            return
        print('\n' + '='*65)
        print('  INFERENCE RESOURCE BREAKDOWN — per prediction path')
        print('='*65)
        hdr = (f'  {"Path":<28}  {"µs/sample":>9}  {"±std µs":>7}  '
               f'{"throughput/s":>13}  {"Peak ΔMem MB":>12}  {"Avg CPU%":>8}  {"MaxThr":>6}')
        print(hdr)
        print('  ' + '-'*92)
        for label, r in self.inference_results.items():
            print(f'  {label:<28}  {r["per_sample_us"]:>9.3f}  '
                  f'{r["per_sample_std_us"]:>7.3f}  '
                  f'{r["throughput_sps"]:>13,.0f}  '
                  f'{r["mem_delta_mb"]:>+12.2f}  '
                  f'{r["avg_cpu_pct"]:>8.1f}  '
                  f'{r["max_threads"]:>6}')

    def print_stage_summary(self):
        df = self.summary_df()
        if df.empty:
            print('  No stages recorded.')
            return
        print('\n' + '='*65)
        print('  STAGE RESOURCE SUMMARY')
        print('='*65)
        print(df.to_string(index=False))

    def print_fdm_vs_ham_delta(self):
        print('\n' + '='*65)
        print('  FDM (4 models) vs HAM (5 models) — inference resource delta')
        print('  ExtraTrees is HAM-only; this shows the cost.')
        print('='*65)
        for tag, fdm_key, ham_key in [
            ('binary',  'FDM  binary',  'HAM  binary'),
            ('3-label', 'FDM  3-label', 'HAM  3-label'),
        ]:
            if fdm_key in self.inference_results and ham_key in self.inference_results:
                fu = self.inference_results[fdm_key]['per_sample_us']
                hu = self.inference_results[ham_key]['per_sample_us']
                print(f'\n  {tag}:')
                print(f'    FDM (4m) : {fu:.3f} µs/sample')
                print(f'    HAM (5m) : {hu:.3f} µs/sample')
                print(f'    Delta    : {hu-fu:+.3f} µs  ({(hu/fu-1)*100:+.1f}%)')

    def print_efficiency_table(self):
        if not self.inference_results:
            return
        print('\n' + '='*65)
        print('  EFFICIENCY METRICS  (higher = better use of resources)')
        print('='*65)
        rows = []
        for label, r in self.inference_results.items():
            us   = r['per_sample_us']
            sps  = r['throughput_sps']
            peak = max(abs(r['mem_delta_mb']), 0.1)
            rows.append({
                'Path'              : label,
                'Samples/s per MB'  : round(sps / peak, 1),
                'Samples/s per CPU%': round(sps / max(r['avg_cpu_pct'], 0.1), 0),
                'µs × Peak ΔMB'    : round(us * peak, 4),
            })
        print(pd.DataFrame(rows).to_string(index=False))

    def print_full_report(self):
        print('\n' + '#'*65)
        print('  FULL INFERENCE RESOURCE REPORT')
        print('#'*65)
        self.print_stage_summary()
        self.print_inference_report()
        self.print_fdm_vs_ham_delta()
        self.print_efficiency_table()

# Alias for backward compat
ResourceMonitor = DetailedResourceMonitor


# ═════════════════════════════════════════════════════════════════════════════
# VMFCVDVoter  (verbatim from training notebook cell 26)
# ═════════════════════════════════════════════════════════════════════════════

class VMFCVDVoter:
    _BASE_MODELS_SPEC = {
        'AdaBoost'        : lambda: AdaBoostClassifier(
            n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE),
        'Bagging'         : lambda: BaggingClassifier(
            n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1),
        'GradientBoosting': lambda: GradientBoostingClassifier(
            n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE),
        'RandomForest'    : lambda: RandomForestClassifier(
            n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1),
    }
    _EXTRATREES_SPEC = {
        'ExtraTrees': lambda: ExtraTreesClassifier(
            n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1),
    }

    def __init__(self, use_class_weights=USE_CLASS_WEIGHTS,
                 use_soft_voting=USE_SOFT_VOTING, use_calibration=USE_CALIBRATION,
                 use_shapley=USE_SHAPLEY_WEIGHTS, include_extratrees=False):
        self.include_extratrees = include_extratrees
        self._models_spec = dict(self._BASE_MODELS_SPEC)
        if include_extratrees:
            self._models_spec['ExtraTrees'] = self._EXTRATREES_SPEC['ExtraTrees']
        self.raw_models = {}
        self.trained_models = {}
        self.model_accuracies = {}
        self.model_weights = {}
        self.TotAccuracy = None
        self.MaxVoteIndex = None
        self.MaxAccuracy = None
        self.disagreement_threshold = None
        self.use_class_weights = use_class_weights
        self.use_soft_voting   = use_soft_voting
        self.use_calibration   = use_calibration
        self.use_shapley       = use_shapley

    def voting_data(self, X):
        VD = np.zeros(len(X), dtype=float)
        for name, model in self.raw_models.items():
            VD += model.predict(X).astype(float) * self.model_weights.get(name, 1.0)
        return VD

    def hard_voting_data(self, X):
        VD = np.zeros(len(X), dtype=float)
        for model in self.raw_models.values():
            VD += model.predict(X).astype(float)
        return VD

    def predict_binary_from_vd(self, VD):
        return (VD >= self.MaxVoteIndex).astype(int)

    def predict_3label(self, X):
        probas = np.stack(
            [m.predict_proba(X)[:, 1] for m in self.trained_models.values()], axis=1)
        mean_p = probas.mean(axis=1)
        std_p  = probas.std(axis=1)
        return np.where(std_p >= self.disagreement_threshold, 1,
               np.where(mean_p >= 0.5, 2, 0))

    def __getstate__(self):
        state = self.__dict__.copy()
        state.pop('_models_spec', None)
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._models_spec = dict(self._BASE_MODELS_SPEC)
        if getattr(self, 'include_extratrees', False):
            self._models_spec['ExtraTrees'] = self._EXTRATREES_SPEC['ExtraTrees']


# ═════════════════════════════════════════════════════════════════════════════
# FastDetectionMode  (verbatim from training notebook cell 28)
# ═════════════════════════════════════════════════════════════════════════════

class FastDetectionMode:
    def __init__(self, fdm_features: list):
        self.fdm_features = fdm_features
        self.voter        = VMFCVDVoter(include_extratrees=False)
        self.trained      = False

    def predict(self, X) -> np.ndarray:
        VD = self.voter.voting_data(X[self.fdm_features])
        return (VD >= self.voter.MaxVoteIndex).astype(int)

    def predict_3label(self, X) -> np.ndarray:
        return self.voter.predict_3label(X[self.fdm_features])

    def evaluate(self, X_test, y_test) -> dict:
        return compute_metrics(np.array(y_test), self.predict(X_test), mode_name='FDM')

    def evaluate_3label(self, X_test, y_test) -> dict:
        return compute_metrics_3label(np.array(y_test),
                                      self.predict_3label(X_test), mode_name='FDM')


# ═════════════════════════════════════════════════════════════════════════════
# DefensiveFastDetectionMode  (verbatim from training notebook cell 30)
# ═════════════════════════════════════════════════════════════════════════════

class DefensiveFastDetectionMode:
    def __init__(self, fdm: FastDetectionMode):
        self.fdm = fdm

    @property
    def trained(self):
        return self.fdm.trained

    def predict(self, X) -> np.ndarray:
        VD = self.fdm.voter.hard_voting_data(X[self.fdm.fdm_features])
        return (VD > 0).astype(int)

    def predict_early_exit(self, X) -> np.ndarray:
        X_feat       = X[self.fdm.fdm_features].reset_index(drop=True)
        n            = len(X_feat)
        results      = np.zeros(n, dtype=int)
        still_benign = np.ones(n, dtype=bool)
        for name in list(self.fdm.voter.raw_models.keys()):
            if not still_benign.any():
                break
            model       = self.fdm.voter.raw_models[name]
            benign_idx  = np.where(still_benign)[0]
            X_remaining = X_feat.iloc[benign_idx].reset_index(drop=True)
            preds       = model.predict(X_remaining)
            flagged     = benign_idx[preds.astype(bool)]
            results[flagged]      = 1
            still_benign[flagged] = False
        return results

    def evaluate(self, X_test, y_test) -> dict:
        return compute_metrics(np.array(y_test), self.predict(X_test), mode_name='DFDM')

    def evaluate_early_exit(self, X_test, y_test) -> dict:
        return compute_metrics(np.array(y_test),
                               self.predict_early_exit(X_test), mode_name='DFDM-EarlyExit')


# ═════════════════════════════════════════════════════════════════════════════
# HighAccuracyMode  (verbatim from training notebook cell 32)
# ═════════════════════════════════════════════════════════════════════════════

class HighAccuracyMode:
    def __init__(self, ham_features: list):
        self.ham_features = ham_features
        self.voter        = VMFCVDVoter(include_extratrees=USE_EXTRATREES_HAM_ONLY)
        self.trained      = False

    def predict(self, X) -> np.ndarray:
        VD = self.voter.voting_data(X[self.ham_features])
        return (VD >= self.voter.MaxVoteIndex).astype(int)

    def predict_3label(self, X) -> np.ndarray:
        return self.voter.predict_3label(X[self.ham_features])

    def evaluate(self, X_test, y_test) -> dict:
        return compute_metrics(np.array(y_test), self.predict(X_test), mode_name='HAM')

    def evaluate_3label(self, X_test, y_test) -> dict:
        return compute_metrics_3label(np.array(y_test),
                                      self.predict_3label(X_test), mode_name='HAM')


# ═════════════════════════════════════════════════════════════════════════════
# VMFCVD controller  (verbatim from training notebook cell 36)
# ═════════════════════════════════════════════════════════════════════════════

class VMFCVD:
    def __init__(self, fdm_features: list, ham_features: list,
                 flow_threshold_high: int   = FLOW_THRESHOLD_HIGH,
                 flow_threshold_extreme: int = FLOW_THRESHOLD_EXTREME):
        self.fdm  = FastDetectionMode(fdm_features)
        self.dfdm = DefensiveFastDetectionMode(self.fdm)
        self.ham  = HighAccuracyMode(ham_features)
        self.flow_threshold_high    = flow_threshold_high
        self.flow_threshold_extreme = flow_threshold_extreme
        self.current_mode  = 'HAM'
        self.trained       = False
        self.smote_applied = False

    def _select_mode(self, network_flow_rate: int) -> str:
        if network_flow_rate >= self.flow_threshold_extreme:
            mode = 'DFDM'
        elif network_flow_rate >= self.flow_threshold_high:
            mode = 'FDM'
        else:
            mode = 'HAM'
        self.current_mode = mode
        return mode

    def predict(self, X, network_flow_rate: int = 0) -> tuple:
        mode = self._select_mode(network_flow_rate)
        if mode == 'DFDM':
            return (self.dfdm.predict_early_exit(X) if USE_EARLY_EXIT
                    else self.dfdm.predict(X)), mode
        elif mode == 'FDM':
            return self.fdm.predict(X), mode
        else:
            return self.ham.predict(X), mode

    def predict_3label(self, X, network_flow_rate: int = 0) -> tuple:
        mode = self._select_mode(network_flow_rate)
        if mode == 'DFDM':
            binary = self.dfdm.predict(X)
            return np.where(binary == 1, 2, 0), mode
        elif mode == 'FDM':
            return self.fdm.predict_3label(X), mode
        else:
            return self.ham.predict_3label(X), mode

    def evaluate_all_modes(self, X_test, y_test) -> dict:
        print('\n' + '='*65 + '\n  VMFCVD -- Binary Evaluation\n' + '='*65)
        return {
            'FDM' : self.fdm.evaluate(X_test, y_test),
            'DFDM': self.dfdm.evaluate(X_test, y_test),
            'HAM' : self.ham.evaluate(X_test, y_test),
        }

    def individual_model_accuracies(self) -> pd.DataFrame:
        rows = []
        for mode_name, voter in [('FDM/DFDM', self.fdm.voter), ('HAM', self.ham.voter)]:
            for model_name, acc in voter.model_accuracies.items():
                rows.append({'Mode': mode_name, 'Model': model_name, 'Accuracy': acc})
        return pd.DataFrame(rows)


# ── Metrics stubs (needed by FastDetectionMode.evaluate etc.) ─────────────────

def compute_metrics(y_true, y_pred, mode_name='') -> dict:
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    TP = int(((y_true==1)&(y_pred==1)).sum())
    TN = int(((y_true==0)&(y_pred==0)).sum())
    FP = int(((y_true==0)&(y_pred==1)).sum())
    FN = int(((y_true==1)&(y_pred==0)).sum())
    total = TP+TN+FP+FN
    acc  = (TP+TN)/total        if total    else 0.
    prec = TP/(TP+FP)           if TP+FP    else 0.
    rec  = TP/(TP+FN)           if TP+FN    else 0.
    f1   = 2*prec*rec/(prec+rec) if prec+rec else 0.
    tag  = f'[{mode_name}] ' if mode_name else ''
    print(f'{tag}Accuracy={acc:.6f}  Precision={prec:.6f}  '
          f'Sensitivity={rec:.6f}  F1={f1:.6f}')
    print(f'{tag}TP={TP}  TN={TN}  FP={FP}  FN={FN}')
    return dict(Accuracy=acc, Precision=prec, Sensitivity=rec, F1=f1,
                TP=TP, TN=TN, FP=FP, FN=FN)


def compute_metrics_3label(y_true, y_pred_3, mode_name='') -> dict:
    y_true   = np.asarray(y_true).flatten()
    y_pred_3 = np.asarray(y_pred_3).flatten()
    n        = len(y_true)
    b = (y_pred_3==0); w = (y_pred_3==1); m = (y_pred_3==2)
    conf = ~w
    conf_acc = 0.
    if conf.sum():
        yb = np.where(y_pred_3==2, 1, 0)
        conf_acc = metrics.accuracy_score(y_true[conf], yb[conf])
    mal_tp   = int(((y_true==1)&m).sum())
    ben_tp   = int(((y_true==0)&b).sum())
    ben_fp   = int(((y_true==1)&b).sum())
    warn_b   = int(((y_true==0)&w).sum())
    warn_m   = int(((y_true==1)&w).sum())
    mal_rec  = mal_tp / max(int((y_true==1).sum()), 1)
    ben_rec  = ben_tp / max(int((y_true==0).sum()), 1)
    ben_prec = ben_tp / max(int(b.sum()), 1)
    mal_prec = mal_tp / max(int(m.sum()), 1)
    wr = w.sum()/n
    tag = f'[{mode_name} 3L] ' if mode_name else ''
    print(f'{tag}ConfidentAcc={conf_acc:.6f}  WarningRate={wr:.4f}  '
          f'MalRecall={mal_rec:.6f}  BenRecall={ben_rec:.6f}')
    print(f'{tag}  Counts: benign={b.sum()}  warning={w.sum()}  malicious={m.sum()}')
    print(f'{tag}  Warnings: {warn_b} benign + {warn_m} malicious')
    print(f'{tag}  AttacksLetThrough={ben_fp}')
    return dict(ConfidentAccuracy=conf_acc, WarningRate=float(wr),
                BenignPrecision=ben_prec, BenignRecall=ben_rec,
                MaliciousPrecision=mal_prec, MaliciousRecall=mal_rec,
                AttacksLetThrough=ben_fp,
                Counts=dict(benign=int(b.sum()), warning=int(w.sum()), malicious=int(m.sum())),
                WarningBreakdown=dict(actually_benign=warn_b, actually_malicious=warn_m))


print('='*65)
print('  Cell 01 complete — all classes and constants defined.')
print('  pickle.load() will now find: VMFCVD, FastDetectionMode,')
print('  DefensiveFastDetectionMode, HighAccuracyMode, VMFCVDVoter')
print('='*65)
print_system_info()


  Cell 01 complete — all classes and constants defined.
  pickle.load() will now find: VMFCVD, FastDetectionMode,
  DefensiveFastDetectionMode, HighAccuracyMode, VMFCVDVoter

  SYSTEM INFORMATION
  CPU cores  (physical) : 2
  CPU cores  (logical)  : 4
  Total RAM             : 31.35 GB
  Available RAM         : 30.27 GB
  RAM utilisation       : 3.4%
  Python                : 3.12.12
  Platform              : linux


## Cell 02 — Configuration
Set these before running any mode cell.

In [2]:
# ── Checkpoint directory ─────────────────────────────────────────────────────
# The folder containing vmfcvd_*_step*.pkl from the training run.
# On Kaggle: typically '/kaggle/working' or '/kaggle/input/<dataset-name>'
CKPT_DIR = '/kaggle/input/datasets/mesbahuddinahamed/vmfcvd-trial-dataset'

# ── Input CSV ─────────────────────────────────────────────────────────────────
CSV_PATH = '/kaggle/input/datasets/mesbahuddinahamed/vmfcvd-trial-dataset/Test.csv'
OUT_PATH = '/kaggle/working/test_predictions.csv'

# ── Label configuration ────────────────────────────────────────────────────────
# Set LABEL_COL to None if your CSV has no ground-truth column → predictions only.
# If labels are strings ('Benign', 'DrDoS_DNS', ...) set BENIGN_LABEL to the benign string.
# If labels are already binary integers (0/1), BENIGN_LABEL is ignored.
LABEL_COL    = 'Label'   # or None
BENIGN_LABEL = 'Benign'

# ── Mode switching ────────────────────────────────────────────────────────────
# None  → all three modes run (Mode C behaviour)
# 0     → HAM  (stable / normal traffic)
# 1000  → FDM  (high volume / DDoS suspected)
# 5000  → DFDM (extreme DDoS, emergency)
FLOW_RATE = None

# ── Single-row values for Mode B ─────────────────────────────────────────────
# Fill these after running Cell 03 to see which features are required.
SINGLE_ROW = {
    'Init Fwd Win Bytes'      : 65535.0,
    'Avg Packet Size'         : 84.5,
    'Fwd Packet Length Max'   : 1460.0,
    'Fwd Packets Length Total': 6840.0,
    'Flow IAT Mean'           : 1200.0,
    'Init Bwd Win Bytes'      : 65535.0,
}

# ── Resource monitor settings ─────────────────────────────────────────────────
N_REPEATS    = 5     # inference repeats for timing (more = more accurate)
BATCH_SIZE   = 4096  # only used in Mode D

print('Configuration loaded.')
print(f'  CKPT_DIR   : {CKPT_DIR}')
print(f'  CSV_PATH   : {CSV_PATH}')
print(f'  LABEL_COL  : {LABEL_COL}')
print(f'  FLOW_RATE  : {FLOW_RATE}  (None = all modes)')
print(f'  N_REPEATS  : {N_REPEATS}')


Configuration loaded.
  CKPT_DIR   : /kaggle/input/datasets/mesbahuddinahamed/vmfcvd-trial-dataset
  CSV_PATH   : /kaggle/input/datasets/mesbahuddinahamed/vmfcvd-trial-dataset/Test.csv
  LABEL_COL  : Label
  FLOW_RATE  : None  (None = all modes)
  N_REPEATS  : 5


## Cell 03 — Load Artifacts
Finds the pkl files, tracks loading time and memory, then verifies the loaded classes.

**This works because Cell 01 already defined all required classes.**
Resource usage during loading is tracked by `INFER_MONITOR`.


In [3]:
def _find_prefix(ckpt_dir):
    pattern = os.path.join(ckpt_dir, 'vmfcvd_*_step10_vmfcvd.pkl')
    matches = glob.glob(pattern)
    if not matches:
        raise FileNotFoundError(
            f'No step10 checkpoint in "{ckpt_dir}".\n'
            f'Run the training pipeline first so it saves vmfcvd_*_step10_vmfcvd.pkl.\n'
            f'Files found in dir: {os.listdir(ckpt_dir) if os.path.isdir(ckpt_dir) else "DIR NOT FOUND"}')
    if len(matches) > 1:
        names = [os.path.basename(m) for m in matches]
        raise RuntimeError(
            f'Multiple step10 checkpoints found: {names}.\n'
            f'Keep only the one you want in CKPT_DIR.')
    fname = os.path.basename(matches[0])
    return os.path.join(ckpt_dir, fname.split('_step10')[0])


def load_artifacts(ckpt_dir=CKPT_DIR):
    # Create a fresh monitor for this session
    global INFER_MONITOR
    INFER_MONITOR = DetailedResourceMonitor(poll_interval=0.05)

    prefix = _find_prefix(ckpt_dir)
    print(f'[ckpt] Using prefix: {os.path.basename(prefix)}')

    # ── vmfcvd object ─────────────────────────────────────────────────────────
    INFER_MONITOR.start('Load_vmfcvd_pkl')
    with open(f'{prefix}_step10_vmfcvd.pkl', 'rb') as f:
        vmfcvd = pickle.load(f)
    INFER_MONITOR.stop('Load_vmfcvd_pkl')
    print(f'[ckpt] vmfcvd      <- {os.path.basename(prefix)}_step10_vmfcvd.pkl')

    # Record serialized sizes of each model group
    INFER_MONITOR.record_model_size('FDM_voter', vmfcvd.fdm.voter)
    INFER_MONITOR.record_model_size('HAM_voter', vmfcvd.ham.voter)

    # ── scaler + encoders ─────────────────────────────────────────────────────
    INFER_MONITOR.start('Load_step03_pkl')
    with open(f'{prefix}_step03_transformed.pkl', 'rb') as f:
        _, scaler, encoders = pickle.load(f)
    INFER_MONITOR.stop('Load_step03_pkl')
    print(f'[ckpt] scaler/enc  <- {os.path.basename(prefix)}_step03_transformed.pkl')

    # ── feature lists ─────────────────────────────────────────────────────────
    INFER_MONITOR.start('Load_step09_pkl')
    with open(f'{prefix}_step09_clusters.pkl', 'rb') as f:
        _, fdm_features, ham_features = pickle.load(f)
    INFER_MONITOR.stop('Load_step09_pkl')
    print(f'[ckpt] features    <- {os.path.basename(prefix)}_step09_clusters.pkl')

    all_features = sorted(set(fdm_features) | set(ham_features))

    # ── Verify ────────────────────────────────────────────────────────────────
    voter_fdm = vmfcvd.fdm.voter
    voter_ham = vmfcvd.ham.voter
    print(f'\n[verify] FDM voter:')
    print(f'  raw_models   : {list(voter_fdm.raw_models.keys())}  ({len(voter_fdm.raw_models)} models)')
    print(f'  MaxVoteIndex : {voter_fdm.MaxVoteIndex:.4f}')
    print(f'  Disagreement : {voter_fdm.disagreement_threshold:.4f}')
    print(f'[verify] HAM voter:')
    print(f'  raw_models   : {list(voter_ham.raw_models.keys())}  ({len(voter_ham.raw_models)} models)')
    print(f'  MaxVoteIndex : {voter_ham.MaxVoteIndex:.4f}')
    print(f'  Disagreement : {voter_ham.disagreement_threshold:.4f}')
    print(f'\n[features] FDM  : {fdm_features}')
    print(f'[features] HAM  : {ham_features}')
    print(f'[features] UNION: {all_features}  ← minimum columns your CSV needs')

    # ── Load timing summary ───────────────────────────────────────────────────
    df_load = INFER_MONITOR.summary_df()
    if not df_load.empty:
        print('\n[resource] Loading stage times:')
        print(df_load[['Stage','Time (s)','Mem Δ (MB)','Peak Δ (MB)','Peak Mem (MB)']].to_string(index=False))

    return dict(vmfcvd=vmfcvd, scaler=scaler, encoders=encoders,
                fdm_features=fdm_features, ham_features=ham_features,
                all_features=all_features)


# ── Run ───────────────────────────────────────────────────────────────────────
ARTS         = load_artifacts()
VMFCVD_MODEL = ARTS['vmfcvd']
SCALER       = ARTS['scaler']
ENCODERS     = ARTS['encoders']
FDM_FEATURES = ARTS['fdm_features']
HAM_FEATURES = ARTS['ham_features']
ALL_FEATURES = ARTS['all_features']


[ckpt] Using prefix: vmfcvd_54ff376019
[ckpt] vmfcvd      <- vmfcvd_54ff376019_step10_vmfcvd.pkl
[ckpt] scaler/enc  <- vmfcvd_54ff376019_step03_transformed.pkl
[ckpt] features    <- vmfcvd_54ff376019_step09_clusters.pkl

[verify] FDM voter:
  raw_models   : ['AdaBoost', 'Bagging', 'GradientBoosting', 'RandomForest']  (4 models)
  MaxVoteIndex : 0.5000
  Disagreement : 0.3455
[verify] HAM voter:
  raw_models   : ['AdaBoost', 'Bagging', 'GradientBoosting', 'RandomForest', 'ExtraTrees']  (5 models)
  MaxVoteIndex : 0.3000
  Disagreement : 0.4150

[features] FDM  : ['Init Fwd Win Bytes', 'Avg Packet Size']
[features] HAM  : ['Init Fwd Win Bytes', 'Fwd Packet Length Max', 'Fwd Packets Length Total', 'Flow IAT Mean', 'Init Bwd Win Bytes']
[features] UNION: ['Avg Packet Size', 'Flow IAT Mean', 'Fwd Packet Length Max', 'Fwd Packets Length Total', 'Init Bwd Win Bytes', 'Init Fwd Win Bytes']  ← minimum columns your CSV needs

[resource] Loading stage times:
          Stage  Time (s)  Mem Δ (MB) 

## Cell 04 — Preprocessing

In [4]:
def preprocess(df_raw, label_col=LABEL_COL, benign_label=BENIGN_LABEL):
    """
    Apply the same preprocessing as the training pipeline.
    Returns (X, y_or_None).
    """
    df = df_raw.copy()
    df.columns = df.columns.str.strip()

    # Label extraction
    y = None
    if label_col and label_col in df.columns:
        raw = df[label_col].copy()
        if raw.dtype == object or str(raw.dtype) == 'category':
            y = (raw != benign_label).astype(int).values
            print(f'[label] Binarized: "{benign_label}"→0, other→1  '
                  f'| benign={int((y==0).sum()):,}  malicious={int((y==1).sum()):,}')
        else:
            y = raw.astype(int).values
            print(f'[label] Numeric 0/1  '
                  f'| benign={int((y==0).sum()):,}  malicious={int((y==1).sum()):,}')
        df.drop(columns=[label_col], inplace=True)
    elif label_col:
        print(f'[label] Column "{label_col}" not in CSV — predictions only.')

    # Validate
    missing = [f for f in ALL_FEATURES if f not in df.columns]
    if missing:
        raise ValueError(
            f'Missing required columns: {missing}\n'
            f'Your CSV must contain at least: {ALL_FEATURES}')

    # Clean
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Encode categoricals
    for col, le in ENCODERS.items():
        if col in df.columns:
            known = set(le.classes_)
            df[col] = df[col].astype(str).apply(
                lambda v: v if v in known else le.classes_[0])
            df[col] = le.transform(df[col])

    # Select features
    X = df[ALL_FEATURES].astype(float)
    X.fillna(0, inplace=True)

    # Scale (StandardScaler with saved training parameters)
    if SCALER is not None:
        scaler_feats = list(SCALER.feature_names_in_)
        cols_to_scale = [c for c in ALL_FEATURES if c in scaler_feats]
        if cols_to_scale:
            idx = [scaler_feats.index(c) for c in cols_to_scale]
            X[cols_to_scale] = ((X[cols_to_scale].values
                                 - SCALER.mean_[idx]) / SCALER.scale_[idx])

    print(f'[preprocess] Ready: {X.shape[0]:,} rows × {X.shape[1]} features')
    return X, y


def single_row_to_df(row_dict):
    missing = [f for f in ALL_FEATURES if f not in row_dict]
    if missing:
        raise ValueError(f'SINGLE_ROW missing columns: {missing}\n'
                         f'Required: {ALL_FEATURES}')
    return pd.DataFrame([row_dict])[ALL_FEATURES]


def _flow_to_mode(flow_rate):
    if flow_rate is None: return None
    if flow_rate >= 5000:  return 'DFDM'
    if flow_rate >= 1000:  return 'FDM'
    return 'HAM'


def _predict_one_mode(X, mode_name, monitor=None):
    """Run binary + 3-label for one mode, with optional resource tracking."""
    fdm_feats = FDM_FEATURES
    ham_feats = HAM_FEATURES
    X_fdm = X[fdm_feats].reset_index(drop=True)
    X_ham = X[ham_feats].reset_index(drop=True)

    if mode_name == 'FDM':
        if monitor:
            r_bin = monitor.record_inference(
                'FDM  binary',  VMFCVD_MODEL.fdm.predict,       X_fdm, N_REPEATS)
            r_3l  = monitor.record_inference(
                'FDM  3-label', VMFCVD_MODEL.fdm.predict_3label, X_fdm, N_REPEATS)
            p_bin = (X_fdm.pipe(lambda x: VMFCVD_MODEL.fdm.voter.voting_data(x))
                     >= VMFCVD_MODEL.fdm.voter.MaxVoteIndex).astype(int)
            p_3l  = VMFCVD_MODEL.fdm.predict_3label(X_fdm)
        else:
            p_bin = VMFCVD_MODEL.fdm.predict(X_fdm)
            p_3l  = VMFCVD_MODEL.fdm.predict_3label(X_fdm)
    elif mode_name == 'DFDM':
        if monitor:
            r_std = monitor.record_inference(
                'DFDM standard',   VMFCVD_MODEL.dfdm.predict,            X, N_REPEATS)
            r_ee  = monitor.record_inference(
                'DFDM early-exit', VMFCVD_MODEL.dfdm.predict_early_exit, X, N_REPEATS)
        p_bin = VMFCVD_MODEL.dfdm.predict_early_exit(X) if USE_EARLY_EXIT else VMFCVD_MODEL.dfdm.predict(X)
        p_3l  = np.where(p_bin == 1, 2, 0)  # DFDM always binary
    else:  # HAM
        if monitor:
            r_bin = monitor.record_inference(
                'HAM  binary',  VMFCVD_MODEL.ham.predict,       X_ham, N_REPEATS)
            r_3l  = monitor.record_inference(
                'HAM  3-label', VMFCVD_MODEL.ham.predict_3label, X_ham, N_REPEATS)
        p_bin = VMFCVD_MODEL.ham.predict(X_ham)
        p_3l  = VMFCVD_MODEL.ham.predict_3label(X_ham)

    return p_bin, p_3l


print('Preprocessing and prediction helpers defined.')


Preprocessing and prediction helpers defined.


## Cell 05 — Inference Modes A / B / C / D

In [5]:
# ── MODE A: CSV ───────────────────────────────────────────────────────────────

def run_csv(csv_path=CSV_PATH, flow_rate=FLOW_RATE,
            label_col=LABEL_COL, benign_label=BENIGN_LABEL,
            save_output=True, track_resources=True):
    """
    Mode A: CSV inference.
    If FLOW_RATE is None, all three modes run (same as Mode C).
    Full binary + 3-label metrics if LABEL_COL is present.
    Resource usage per inference path is tracked when track_resources=True.
    """
    sep = '='*65
    print(sep); print(f'  MODE A — CSV inference: {csv_path}'); print(sep)

    df_raw = pd.read_csv(csv_path)
    print(f'[csv] {len(df_raw):,} rows  {df_raw.shape[1]} columns')

    INFER_MONITOR.start('Preprocess')
    X, y = preprocess(df_raw, label_col, benign_label)
    INFER_MONITOR.stop('Preprocess')

    mode     = _flow_to_mode(flow_rate)
    modes    = [mode] if mode else ['HAM', 'FDM', 'DFDM']
    monitor  = INFER_MONITOR if track_resources else None

    label3   = {0:'Benign', 1:'Warning', 2:'Malicious'}
    all_res  = {}

    for m in modes:
        print(f'\n{sep}\n  Predictions — {m}\n{sep}')
        t0 = time.perf_counter()
        p_bin, p_3l = _predict_one_mode(X, m, monitor=monitor)
        us = (time.perf_counter()-t0)/len(X)*1e6
        n_mal = int((p_bin==1).sum())
        print(f'[{m}] Benign={len(p_bin)-n_mal:,}  Malicious={n_mal:,}  '
              f'({n_mal/len(p_bin)*100:.2f}% attack)  {us:.2f} µs/sample (wall)')
        if m != 'DFDM':
            n_w = int((p_3l==1).sum())
            print(f'[{m} 3L] Benign={int((p_3l==0).sum())}  Warning={n_w}  '
                  f'Malicious={int((p_3l==2).sum())}  '
                  f'(warn_rate={n_w/len(p_3l)*100:.2f}%)')
        res = {'predictions_binary': p_bin, 'predictions_3label': p_3l}
        if y is not None:
            print('\nBinary metrics:')
            res['metrics_binary'] = compute_metrics(y, p_bin, mode_name=m)
            if m != 'DFDM':
                print('3-label metrics:')
                res['metrics_3label'] = compute_metrics_3label(y, p_3l, mode_name=m)
        all_res[m] = res

    # Build output DataFrame
    df_out = df_raw.copy()
    for m, r in all_res.items():
        sfx = '' if len(all_res)==1 else f'_{m}'
        df_out[f'pred_binary{sfx}']  = r['predictions_binary']
        df_out[f'pred_3label{sfx}']  = pd.Series(r['predictions_3label']).map(label3).values
        df_out[f'mode{sfx}']         = m

    if save_output:
        df_out.to_csv(OUT_PATH, index=False)
        print(f'\n[save] {OUT_PATH}')

    return all_res, df_out


# ── MODE B: SINGLE ROW ────────────────────────────────────────────────────────

def run_single_row(row_dict=SINGLE_ROW, flow_rate=FLOW_RATE, true_label=None):
    """
    Mode B: single-row inference.
    No metrics unless true_label is provided (0 or 1).
    All three modes run when FLOW_RATE=None.
    """
    sep = '='*65
    print(sep); print('  MODE B — Single row inference'); print(sep)

    df_single = single_row_to_df(row_dict)
    X, _ = preprocess(df_single, label_col=None)

    mode  = _flow_to_mode(flow_rate)
    modes = [mode] if mode else ['HAM', 'FDM', 'DFDM']
    label_b = {0:'Benign', 1:'Malicious'}
    label_3 = {0:'Benign', 1:'Warning',  2:'Malicious'}

    for m in modes:
        p_bin, p_3l = _predict_one_mode(X, m, monitor=None)
        pred_b = label_b[int(p_bin[0])]
        pred_3 = label_3[int(p_3l[0])]
        correct = ''
        if true_label is not None:
            exp = label_b[int(true_label)]
            correct = ('  ✓ correct' if int(p_bin[0])==int(true_label)
                       else f'  ✗ wrong (expected {exp})')
        print(f'\n  [{m}]')
        print(f'    Binary  : {pred_b}{correct}')
        if m != 'DFDM':
            print(f'    3-label : {pred_3}', end='')
            if pred_3 == 'Warning':
                print('  ← models disagree (high probability std)', end='')
            print()
        else:
            print(f'    DFDM is always binary (no warning zone in emergency mode)')

    print(f'\n  Feature values used:')
    for col in ALL_FEATURES:
        print(f'    {col:35s}: {row_dict.get(col, "MISSING")}')


# ── MODE C: ALL MODES COMPARISON ──────────────────────────────────────────────

def run_all_modes(csv_path=OUT_PATH, label_col=LABEL_COL,
                  benign_label=BENIGN_LABEL, track_resources=True):
    """
    Mode C: forces HAM + FDM + DFDM regardless of FLOW_RATE.
    Prints a summary comparison table.
    """
    sep = '='*65
    print(sep); print('  MODE C — All modes comparison (HAM / FDM / DFDM)'); print(sep)

    df_raw = pd.read_csv(csv_path)
    print(f'[csv] {len(df_raw):,} rows')
    X, y = preprocess(df_raw, label_col, benign_label)
    monitor = INFER_MONITOR if track_resources else None

    summary = []
    for m in ['HAM', 'FDM', 'DFDM']:
        t0 = time.perf_counter()
        p_bin, p_3l = _predict_one_mode(X, m, monitor=monitor)
        us = (time.perf_counter()-t0)/len(X)*1e6
        row = {'Mode': m, 'Speed_us(wall)': round(us, 2),
               'Malicious%': round(p_bin.mean()*100, 2)}
        if y is not None:
            mb = compute_metrics(y, p_bin, mode_name=m)
            row.update({k: round(v, 6) for k, v in mb.items()
                        if k in ('Accuracy','Precision','Sensitivity','F1')})
            if m != 'DFDM':
                m3 = compute_metrics_3label(y, p_3l, mode_name=m)
                row['ConfAcc']       = round(m3['ConfidentAccuracy'], 6)
                row['WarnRate%']     = round(m3['WarningRate']*100, 3)
                row['AttacksMissed'] = m3['AttacksLetThrough']
        summary.append(row)

    print('\n' + '='*65 + '\n  SUMMARY TABLE\n' + '='*65)
    print(pd.DataFrame(summary).set_index('Mode').to_string())
    return pd.DataFrame(summary)


# ── MODE D: BATCH PREDICTOR ───────────────────────────────────────────────────

def run_batch_predictor(csv_path=CSV_PATH, flow_rate=FLOW_RATE,
                        batch_size=BATCH_SIZE, label_col=LABEL_COL,
                        benign_label=BENIGN_LABEL):
    """
    Mode D: mini-batch inference for files too large for direct predict().
    SLOWER than Mode A for normal-sized files.  Use only to avoid OOM.
    FLOW_RATE must be a specific integer (not None) — one mode only.
    """
    if flow_rate is None:
        raise ValueError('Set FLOW_RATE to 0, 1000, or 5000 for batch mode.')
    mode = _flow_to_mode(flow_rate)
    print(f'  MODE D — Batch predictor  mode={mode}  batch_size={batch_size}')
    print('  NOTE: Use Mode A for normal files — it is faster.')

    reader   = pd.read_csv(csv_path, chunksize=batch_size)
    all_bin, all_3l, all_y = [], [], []
    n_total = 0
    label3  = {0:'Benign', 1:'Warning', 2:'Malicious'}

    for i, chunk in enumerate(reader):
        X_chunk, y_chunk = preprocess(chunk, label_col, benign_label)
        p_bin, p_3l = _predict_one_mode(X_chunk, mode, monitor=None)
        all_bin.append(p_bin); all_3l.append(p_3l)
        if y_chunk is not None: all_y.append(y_chunk)
        n_total += len(X_chunk)
        if i % 10 == 0:
            print(f'  Batch {i+1:>4d}  processed={n_total:,}')

    all_bin = np.concatenate(all_bin)
    all_3l  = np.concatenate(all_3l)
    print(f'\n[done] {n_total:,} rows  Benign={int((all_bin==0).sum())}  '
          f'Malicious={int((all_bin==1).sum())}')

    results = {'predictions_binary': all_bin, 'predictions_3label': all_3l}
    if all_y:
        y_all = np.concatenate(all_y)
        print('\nBinary metrics:')
        results['metrics_binary'] = compute_metrics(y_all, all_bin, mode_name=f'{mode}-batch')
        if mode != 'DFDM':
            print('3-label metrics:')
            results['metrics_3label'] = compute_metrics_3label(y_all, all_3l, mode_name=f'{mode}-batch')
    return results


print('All mode functions defined.')
print('  run_csv()            → Mode A  (CSV inference)')
print('  run_single_row()     → Mode B  (single row)')
print('  run_all_modes()      → Mode C  (HAM+FDM+DFDM comparison)')
print('  run_batch_predictor()→ Mode D  (large files only)')


All mode functions defined.
  run_csv()            → Mode A  (CSV inference)
  run_single_row()     → Mode B  (single row)
  run_all_modes()      → Mode C  (HAM+FDM+DFDM comparison)
  run_batch_predictor()→ Mode D  (large files only)


## Run Mode A — CSV Inference
Edit `CSV_PATH` and `FLOW_RATE` in Cell 02 first.

In [6]:
results_A, df_A = run_csv()


  MODE A — CSV inference: /kaggle/input/datasets/mesbahuddinahamed/vmfcvd-trial-dataset/Test.csv
[csv] 12,462 rows  78 columns
[label] Binarized: "Benign"→0, other→1  | benign=2,042  malicious=10,420
[preprocess] Ready: 12,462 rows × 6 features

  Predictions — HAM
[HAM] Benign=2,042  Malicious=10,420  (83.61% attack)  414.91 µs/sample (wall)
[HAM 3L] Benign=2042  Warning=3  Malicious=10417  (warn_rate=0.02%)

Binary metrics:
[HAM] Accuracy=1.000000  Precision=1.000000  Sensitivity=1.000000  F1=1.000000
[HAM] TP=10420  TN=2042  FP=0  FN=0
3-label metrics:
[HAM 3L] ConfidentAcc=1.000000  WarningRate=0.0002  MalRecall=0.999712  BenRecall=1.000000
[HAM 3L]   Counts: benign=2042  warning=3  malicious=10417
[HAM 3L]   Warnings: 0 benign + 3 malicious
[HAM 3L]   AttacksLetThrough=0

  Predictions — FDM
[FDM] Benign=2,042  Malicious=10,420  (83.61% attack)  204.30 µs/sample (wall)
[FDM 3L] Benign=2030  Warning=18  Malicious=10414  (warn_rate=0.14%)

Binary metrics:
[FDM] Accuracy=0.999037  Pr

## Run Mode B — Single Row
Edit `SINGLE_ROW` in Cell 02 first.

In [7]:
run_single_row()
# To verify against a known label:
# run_single_row(true_label=1)


  MODE B — Single row inference
[preprocess] Ready: 1 rows × 6 features

  [HAM]
    Binary  : Benign
    3-label : Benign

  [FDM]
    Binary  : Benign
    3-label : Benign

  [DFDM]
    Binary  : Benign
    DFDM is always binary (no warning zone in emergency mode)

  Feature values used:
    Avg Packet Size                    : 84.5
    Flow IAT Mean                      : 1200.0
    Fwd Packet Length Max              : 1460.0
    Fwd Packets Length Total           : 6840.0
    Init Bwd Win Bytes                 : 65535.0
    Init Fwd Win Bytes                 : 65535.0


## Run Mode C — All Modes Comparison
Runs all three modes regardless of `FLOW_RATE`.

In [8]:
summary_C = run_all_modes()


  MODE C — All modes comparison (HAM / FDM / DFDM)
[csv] 12,462 rows
[label] Binarized: "Benign"→0, other→1  | benign=2,042  malicious=10,420
[preprocess] Ready: 12,462 rows × 6 features
[HAM] Accuracy=1.000000  Precision=1.000000  Sensitivity=1.000000  F1=1.000000
[HAM] TP=10420  TN=2042  FP=0  FN=0
[HAM 3L] ConfidentAcc=1.000000  WarningRate=0.0002  MalRecall=0.999712  BenRecall=1.000000
[HAM 3L]   Counts: benign=2042  warning=3  malicious=10417
[HAM 3L]   Warnings: 0 benign + 3 malicious
[HAM 3L]   AttacksLetThrough=0
[FDM] Accuracy=0.999037  Precision=0.999424  Sensitivity=0.999424  F1=0.999424
[FDM] TP=10414  TN=2036  FP=6  FN=6
[FDM 3L] ConfidentAcc=0.999437  WarningRate=0.0014  MalRecall=0.999136  BenRecall=0.992165
[FDM 3L]   Counts: benign=2030  warning=18  malicious=10414
[FDM 3L]   Warnings: 13 benign + 5 malicious
[FDM 3L]   AttacksLetThrough=4
[DFDM] Accuracy=0.997833  Precision=0.997797  Sensitivity=0.999616  F1=0.998706
[DFDM] TP=10416  TN=2019  FP=23  FN=4

  SUMMARY TA

## Resource Report
Prints the full `DetailedResourceMonitor` report: loading times, preprocessing,
inference per path (µs/sample, throughput, peak memory, CPU%), FDM vs HAM delta,
and efficiency table.

Run this after any mode cell to see resource usage for that session.


In [9]:
# Stop the background polling thread before printing
# (safe to call multiple times)
try:
    INFER_MONITOR.stop_all_polling()
except Exception:
    pass

INFER_MONITOR.print_full_report()



#################################################################
  FULL INFERENCE RESOURCE REPORT
#################################################################

  STAGE RESOURCE SUMMARY
          Stage  Time (s)  Mem Δ (MB)  Peak Δ (MB)  Peak Mem (MB)  Avg CPU (%)  Max CPU (%)  Max Threads  Poll samples
Load_vmfcvd_pkl     0.470      109.43       106.15         354.44         34.8         58.7           23            10
Load_step03_pkl     1.243      202.02       195.37         568.20         24.8         81.8           23            25
Load_step09_pkl     0.019     -197.34         0.00         574.85          0.0          0.0           23             1
     Preprocess     0.008        0.03         0.00         401.20          0.0          0.0           23             1

  INFERENCE RESOURCE BREAKDOWN — per prediction path
  Path                          µs/sample  ±std µs   throughput/s  Peak ΔMem MB  Avg CPU%  MaxThr
  -----------------------------------------------------------

## Run Mode D — Batch Predictor *(Optional — large files only)*

In [10]:
# Uncomment only when your CSV is too large to load all at once.
# Set FLOW_RATE to a specific value (not None) before running.
#results_D = run_batch_predictor()
print('Mode D defined. Uncomment the line above to run.')


Mode D defined. Uncomment the line above to run.
